# MILP Course Scheduling — Practice Notebook

**Goal:** Build a Mixed-Integer Linear Program with Pyomo that assigns every
course to exactly one timeslot and one room, satisfying all scheduling constraints
and minimising total cost.

---

## Structure

| Part | Content | Status |
|------|---------|--------|
| 1 | Decision variables | Given |
| 2 | Hard constraints C1–C5 | **You complete** |
| 2.6 | Hard constraint C6 (optional) | **You complete** |
| 3 | Objective function | **You complete** |
| 4 | Solve and display results | Given |
| 5 | Soft constraints extension | **You complete** |
| 6 | Visualisation | **You complete** (data extraction only) |

---

### Pyomo quick reference

```python
# Rule-based constraint
def my_rule(m, i, j):
    return expression <= 1
model.C_name = pyo.Constraint(model.I, model.J, rule=my_rule)

# Skip infeasible indices inside a rule
    return pyo.Constraint.Skip

# ConstraintList (for irregular pair sets)
model.CL = pyo.ConstraintList()
model.CL.add(expr <= 1)

# Read a variable value after solving
pyo.value(model.x[c, t, r])
```


In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

try:
    import pyomo.environ as pyo
    print("Pyomo ready.")
except ImportError:
    print("Install Pyomo first:  pip install pyomo")
    raise

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
sns.set_theme(style="whitegrid")


In [ ]:
DATA = "data_teaching"

courses   = pd.read_csv(f"{DATA}/courses.csv")
rooms     = pd.read_csv(f"{DATA}/rooms.csv")
timeslots = pd.read_csv(f"{DATA}/timeslots.csv")
lecturers = pd.read_csv(f"{DATA}/lecturers.csv")
conflicts = pd.read_csv(f"{DATA}/course_conflicts.csv")
options   = pd.read_csv(f"{DATA}/course_period_options.csv")

print("Data loaded.")
for name, df in [("courses", courses), ("rooms", rooms), ("timeslots", timeslots),
                 ("conflicts", conflicts), ("options", options)]:
    print(f"  {name:<12} {df.shape}")


In [ ]:
# -- Sets (Python lists used as Pyomo Set initializers) ---------------
C = list(courses["course_id"].astype(int))
T = list(timeslots["timeslot_id"].astype(int))
R = list(rooms["room_id"].astype(int))

# -- Parameters -------------------------------------------------------
room_cap    = dict(zip(rooms["room_id"].astype(int),      rooms["capacity"]))
room_cost   = dict(zip(rooms["room_id"].astype(int),      rooms["cost_per_slot"]))
room_name   = dict(zip(rooms["room_id"].astype(int),      rooms["name"]))
course_cap  = dict(zip(courses["course_id"].astype(int),  courses["capacity_needed"]))
course_stud = dict(zip(courses["course_id"].astype(int),  courses["n_students"]))
course_name = dict(zip(courses["course_id"].astype(int),  courses["name"]))
course_pref = dict(zip(courses["course_id"].astype(int),  courses["preferred_period"]))
slot_cost   = dict(zip(timeslots["timeslot_id"].astype(int), timeslots["cost"]))
slot_period = dict(zip(timeslots["timeslot_id"].astype(int), timeslots["period"]))
slot_day    = dict(zip(timeslots["timeslot_id"].astype(int), timeslots["day"]))

# -- Conflict edge list (C4) ------------------------------------------
conflict_pairs = [(int(row["course_a"]), int(row["course_b"]))
                  for _, row in conflicts.iterrows()]

# -- Lecturer pair list (C5) ------------------------------------------
lecturer_pairs = []
for _, grp in courses.groupby("lecturer_id"):
    ids = list(grp["course_id"].astype(int))
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            lecturer_pairs.append((ids[i], ids[j]))

# -- Period restriction map (C6) --------------------------------------
allowed_periods = {}
for cid in options["course_id"].unique():
    cid_int = int(cid)
    allowed_periods[cid_int] = set(options[options["course_id"] == cid]["period"])
restricted_courses = list(allowed_periods.keys())

print(f"|C| = {len(C)}   |T| = {len(T)}   |R| = {len(R)}")
print(f"Student conflict pairs (C4) : {len(conflict_pairs)}")
print(f"Lecturer conflict pairs (C5): {len(lecturer_pairs)}")
print(f"Restricted courses      (C6): {restricted_courses}")


---
## Part 1 — Decision Variables

One binary variable per (course, timeslot, room) triple:

$$x_{c,t,r} \in \{0,1\} \qquad \forall\, c \in C,\; t \in T,\; r \in R$$

$x_{c,t,r} = 1$ means course $c$ is scheduled in timeslot $t$ and room $r$.

> This cell is **complete** — read it carefully before continuing.


In [ ]:
model = pyo.ConcreteModel("CourseScheduling")

model.C = pyo.Set(initialize=C)
model.T = pyo.Set(initialize=T)
model.R = pyo.Set(initialize=R)

model.x = pyo.Var(model.C, model.T, model.R, domain=pyo.Binary)

n_vars = len(C) * len(T) * len(R)
print(f"Model     : {model.name}")
print(f"Variables : {n_vars}  binary  model.x[course_id, timeslot_id, room_id]")
print(f"Example   : model.x[1, 2, 3] = 1  means course 1 in timeslot 2, room 3")


---
## Part 2 — Hard Constraints

| # | Name | Mathematical form |
|---|------|-------------------|
| C1 | Assignment | $\sum_{t,r} x_{c,t,r} = 1 \quad \forall c$ |
| C2 | Room conflict | $\sum_{c} x_{c,t,r} \leq 1 \quad \forall t, r$ |
| C3 | Capacity | $x_{c,t,r} = 0$ when $\text{cap}(r) < \text{size}(c)$ |
| C4 | Student clash | $\sum_r x_{c_1,t,r} + \sum_r x_{c_2,t,r} \leq 1 \quad \forall (c_1,c_2)\in E, \forall t$ |
| C5 | Lecturer clash | same form as C4, using `lecturer_pairs` |
| C6 | Period restriction | $x_{c,t,r}=0$ when $\text{period}(t) \notin A_c$ |


In [ ]:
# -- C1: Each course is assigned exactly once -------------------------
#
#   sum_{t in T, r in R}  x[c,t,r]  =  1    for all c in C
#
# Pyomo pattern:
#   def c1_rule(m, c):
#       return sum(m.x[c, t, r] for t in m.T for r in m.R) == 1
#   model.C1 = pyo.Constraint(model.C, rule=c1_rule)
# ---------------------------------------------------------------------

# TODO: define c1_rule and attach it as model.C1
pass

print("C1 added." if hasattr(model, "C1") else "C1 not yet added.")


In [ ]:
# -- C2: No two courses share the same (timeslot, room) ---------------
#
#   sum_{c in C}  x[c,t,r]  <=  1    for all t in T, r in R
#
# Pyomo pattern:
#   def c2_rule(m, t, r):
#       return sum(m.x[c, t, r] for c in m.C) <= 1
#   model.C2 = pyo.Constraint(model.T, model.R, rule=c2_rule)
# ---------------------------------------------------------------------

# TODO: define c2_rule and attach it as model.C2
pass

print("C2 added." if hasattr(model, "C2") else "C2 not yet added.")


In [ ]:
# -- C3: Room capacity >= class size ----------------------------------
#
#   x[c,t,r] = 0   whenever  room_cap[r] < course_cap[c]
#
# Use pyo.Constraint.Skip to leave feasible triples unconstrained:
#
#   def c3_rule(m, c, t, r):
#       if room_cap[r] < course_cap[c]:
#           return m.x[c, t, r] == 0
#       return pyo.Constraint.Skip
#   model.C3 = pyo.Constraint(model.C, model.T, model.R, rule=c3_rule)
# ---------------------------------------------------------------------

# TODO: define c3_rule and attach it as model.C3
pass

if hasattr(model, "C3"):
    n_fixed = sum(1 for _ in model.C3)
    print(f"C3 added — {n_fixed} infeasible (c,t,r) triples fixed to 0")
    eligible = [room_name[r] for r in R if room_cap[r] >= course_cap[3]]
    print(f"Example: {course_name[3]} (needs {course_cap[3]} seats) -> eligible: {eligible}")
else:
    print("C3 not yet added.")


In [ ]:
# -- C4: Courses sharing students cannot be at the same timeslot ------
#
#   sum_r x[c1,t,r] + sum_r x[c2,t,r]  <=  1
#   for all (c1,c2) in conflict_pairs, for all t in T
#
# Use a ConstraintList — the pairs are irregular edges.
#
#   model.C4 = pyo.ConstraintList()
#   for c1, c2 in conflict_pairs:
#       for t in T:
#           model.C4.add(
#               sum(model.x[c1, t, r] for r in R)
#               + sum(model.x[c2, t, r] for r in R) <= 1
#           )
# ---------------------------------------------------------------------

# TODO: create model.C4 as a ConstraintList and populate it
pass

if hasattr(model, "C4"):
    n_c4 = sum(1 for _ in model.C4)
    print(f"C4 added — {n_c4} constraints  (expected {len(conflict_pairs) * len(T)})")
else:
    print("C4 not yet added.")


In [ ]:
# -- C5: A lecturer cannot teach two courses at the same timeslot -----
#
#   sum_r x[c1,t,r] + sum_r x[c2,t,r]  <=  1
#   for all (c1,c2) in lecturer_pairs, for all t in T
#
# Same structure as C4 — use a ConstraintList over lecturer_pairs.
# Binding pairs: Prof. Le (courses 3+6) and Prof. Nguyen (courses 1+9).
# ---------------------------------------------------------------------

# TODO: create model.C5 as a ConstraintList and populate it
pass

if hasattr(model, "C5"):
    n_c5 = sum(1 for _ in model.C5)
    print(f"C5 added — {n_c5} constraints  (expected {len(lecturer_pairs) * len(T)})")
else:
    print("C5 not yet added.")


In [ ]:
# -- C6 (optional): Period restrictions --------------------------------
#
#   x[c,t,r] = 0   if  slot_period[t]  not in  allowed_periods[c]
#   Only for restricted_courses = [6, 10]
#
#   allowed_periods[6]  = {"P3","P4","P5"}  (Databases lab, from 09:30)
#   allowed_periods[10] = {"P1","P2","P3"}  (ML Basics, morning only)
#
# Hint:
#   model.C6 = pyo.ConstraintList()
#   for c in restricted_courses:
#       for t in T:
#           if slot_period[t] not in allowed_periods[c]:
#               for r in R:
#                   model.C6.add(model.x[c, t, r] == 0)
# ---------------------------------------------------------------------

# TODO: implement C6
pass

if hasattr(model, "C6"):
    n_c6 = sum(1 for _ in model.C6)
    print(f"C6 added — {n_c6} variables fixed to 0")
else:
    print("C6 not added (optional).")


---
## Part 3 — Objective Function

Minimise the total scheduling cost:

$$\min \sum_{c \in C}\sum_{t \in T}\sum_{r \in R} \bigl(\text{rc}[r] + \text{tc}[t]\bigr) \cdot x_{c,t,r}$$

| Parameter | Source | Range |
|-----------|--------|-------|
| `room_cost[r]` | `rooms.cost_per_slot` | 1 – 5 |
| `slot_cost[t]` | `timeslots.cost` | 1 – 3 |


In [ ]:
# -- Objective: minimise total cost -----------------------------------
#
#   min  sum_{c,t,r}  (room_cost[r] + slot_cost[t])  *  x[c,t,r]
#
# Pyomo pattern:
#   def obj_rule(m):
#       return sum(
#           (room_cost[r] + slot_cost[t]) * m.x[c, t, r]
#           for c in m.C for t in m.T for r in m.R
#       )
#   model.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)
# ---------------------------------------------------------------------

# TODO: define obj_rule and attach it as model.obj
pass

print("Objective set." if hasattr(model, "obj") else "No objective yet — complete the TODO above.")


---
## Part 4 — Solve and Display Results

> This section is **complete**. Run it once Parts 2 and 3 are done.

**Solver:** HiGHS via `highspy`.  
Install: `pip install highspy`


In [ ]:
solver = pyo.SolverFactory("appsi_highs")

if not solver.available():
    print("Solver not found. Install HiGHS: pip install highspy")
    sol = pd.DataFrame()
else:
    result = solver.solve(model, tee=False)

    is_optimal = result.termination_condition == pyo.TerminationCondition.optimal
    status = "Optimal" if is_optimal else str(result.termination_condition)
    print(f"Solver status : {status}")

    if not is_optimal:
        print()
        print("No optimal solution found.")
        print("Checklist:")
        print("  1. Did you add ALL constraints in Part 2?")
        print("  2. Did you add the objective in Part 3?")
        print("  3. Re-run cells from Part 1 downward to rebuild the model.")
        sol = pd.DataFrame()
    else:
        opt_cost = pyo.value(model.obj)
        print(f"Optimal cost  : {opt_cost:.0f}")
        print()

        rows = []
        for c in C:
            for t in T:
                for r in R:
                    val = pyo.value(model.x[c, t, r])
                    if val is not None and val > 0.5:
                        rows.append({
                            "course_id"  : c,
                            "course"     : course_name[c],
                            "day"        : slot_day[t],
                            "period"     : slot_period[t],
                            "timeslot_id": t,
                            "room_id"    : r,
                            "room"       : room_name[r],
                            "enrolled"   : course_stud[c],
                            "capacity"   : room_cap[r],
                            "cost"       : room_cost[r] + slot_cost[t],
                        })

        day_order    = ["Mon", "Tue", "Wed"]
        period_order = ["P1", "P2", "P3", "P4", "P5"]
        sol = pd.DataFrame(rows)
        sol["day"]    = sol["day"].astype(pd.CategoricalDtype(day_order, ordered=True))
        sol["period"] = sol["period"].astype(pd.CategoricalDtype(period_order, ordered=True))
        sol = sol.sort_values(["day", "period"]).reset_index(drop=True)
        print(f"Assignments  : {len(sol)} (should equal {len(C)})")


In [ ]:
if "sol" in dir() and not sol.empty:
    display(sol[["course", "day", "period", "room", "enrolled", "capacity", "cost"]])

    print()
    print("Schedule pivot (course / room):")
    sol["entry"] = sol["course"].str[:10] + " / " + sol["room"].str[:6]
    pivot = (
        sol.groupby(["period", "day"])["entry"]
           .apply(lambda s: "  |  ".join(s))
           .unstack("day")
           .reindex(columns=["Mon", "Tue", "Wed"])
    )
    print(pivot.to_string())
    print()
    total = sol["cost"].sum()
    print(f"Total cost: {total}")


---
## Part 5 — Soft Constraints Extension

Each course has a `preferred_period`. Add a preference penalty to the objective:

$$\min \sum_{c,t,r} \bigl(\text{rc}[r] + \text{tc}[t] + \lambda \cdot \text{pen}_{c,t}\bigr) \cdot x_{c,t,r}$$

where $\text{pen}_{c,t} = 1$ if $\text{period}(t) \neq \text{preferred\_period}(c)$, else $0$.

Rebuild the model with this objective and experiment with $\lambda \in \{0, 1, 2, 5\}$.


In [ ]:
# Preference penalty dict
pen = {(c, t): 0 if slot_period[t] == course_pref[c] else 1
       for c in C for t in T}

lam = 2   # penalty weight — try 0, 1, 2, 5

# Build a new model with the soft objective
model_soft = pyo.ConcreteModel("CourseScheduling_soft")
model_soft.C = pyo.Set(initialize=C)
model_soft.T = pyo.Set(initialize=T)
model_soft.R = pyo.Set(initialize=R)
model_soft.x = pyo.Var(model_soft.C, model_soft.T, model_soft.R, domain=pyo.Binary)

# TODO (A): add all hard constraints to model_soft
#   Copy each block from Part 2, replacing model with model_soft.
pass

# TODO (B): add the soft objective
#   def soft_obj_rule(m):
#       return sum(
#           (room_cost[r] + slot_cost[t] + lam * pen[(c, t)]) * m.x[c, t, r]
#           for c in m.C for t in m.T for r in m.R
#       )
#   model_soft.obj = pyo.Objective(rule=soft_obj_rule, sense=pyo.minimize)
pass

# Solve
if "solver" in dir() and solver.available() and hasattr(model_soft, "obj"):
    result_soft = solver.solve(model_soft, tee=False)
    ok_soft = result_soft.termination_condition == pyo.TerminationCondition.optimal
    label = "Optimal" if ok_soft else str(result_soft.termination_condition)
    print(f"Soft model status : {label}")
    if ok_soft:
        print(f"Soft objective    : {pyo.value(model_soft.obj):.0f}  (lambda={lam})")
        pref_met = sum(
            1 for c in C for t in T for r in R
            if pyo.value(model_soft.x[c, t, r]) is not None
            and pyo.value(model_soft.x[c, t, r]) > 0.5
            and slot_period[t] == course_pref[c]
        )
        print(f"Preferences met   : {pref_met} / {len(C)} courses")
else:
    print("Complete the TODOs above, then re-run.")


---
## Part 6 — Visualisation

Two charts:
1. **Weekly schedule grid** — which course is in which (day, period) cell, coloured by room
2. **Room utilisation** — enrolled students vs room capacity per assignment

The drawing code is **provided**. Complete only **Step 1** (building the `schedule` dict).


In [ ]:
# -- Step 1: Build the schedule dict (TODO) ---------------------------
#
# schedule: (day, period) -> list of (course_name, room_name, room_id)
#
# Hint:
#   for c in C:
#       for t in T:
#           for r in R:
#               val = pyo.value(model.x[c, t, r])
#               if val is not None and val > 0.5:
#                   key = (slot_day[t], slot_period[t])
#                   if key not in schedule:
#                       schedule[key] = []
#                   schedule[key].append((course_name[c], room_name[r], r))

schedule = {}
# TODO: populate schedule from the solution
pass

# -- Step 2: Draw the grid (given — do not modify) -------------------
if not schedule:
    print("schedule is empty — complete Step 1 first, then re-run.")
else:
    days    = ["Mon", "Tue", "Wed"]
    periods = ["P1", "P2", "P3", "P4", "P5"]
    pt_lbl  = ["07:30", "08:30", "09:30", "10:30", "13:30"]
    palette  = ["#3498db", "#2ecc71", "#e67e22", "#e74c3c", "#9b59b6"]
    room_clr = dict(zip(R, palette))

    fig, axes = plt.subplots(len(days), 1, figsize=(14, 2.6 * len(days)), sharex=True)
    for di, day in enumerate(days):
        ax = axes[di]
        ax.set_xlim(-0.5, len(periods) - 0.5)
        ax.set_ylim(0, 1)
        ax.set_yticks([])
        ax.set_ylabel(day, rotation=0, labelpad=35, va="center", fontweight="bold")
        ax.set_facecolor("#f8f9fa")
        for j in range(len(periods)):
            ax.axvline(j - 0.5, color="lightgray", linewidth=0.8)
        if di == len(days) - 1:
            ax.set_xticks(range(len(periods)))
            ax.set_xticklabels([f"{p}\n{s}" for p, s in zip(periods, pt_lbl)])
        for j, period in enumerate(periods):
            entries = schedule.get((day, period), [])
            for k, (cname, rname, rid) in enumerate(entries):
                color  = room_clr.get(rid, "#bdc3c7")
                y_base = 0.82 - k * 0.42
                ax.add_patch(plt.Rectangle(
                    (j - 0.44, y_base - 0.17), 0.88, 0.34,
                    facecolor=color, alpha=0.78, edgecolor="white", linewidth=1.5
                ))
                short = cname[:14] + ".." if len(cname) > 14 else cname
                ax.text(j, y_base + 0.02, short,
                        ha="center", va="center", fontsize=7.5, fontweight="bold")
                ax.text(j, y_base - 0.1, rname,
                        ha="center", va="center", fontsize=6.5, color="#2c3e50")

    legend_patches = [mpatches.Patch(facecolor=room_clr[r], label=room_name[r],
                                     edgecolor="white") for r in R]
    fig.legend(handles=legend_patches, loc="upper right",
               title="Room", fontsize=9, bbox_to_anchor=(1.01, 0.98))
    fig.suptitle("Weekly Schedule", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


In [ ]:
# -- Room utilisation chart -------------------------------------------
if "sol" not in dir() or sol.empty:
    print("No solution yet — run Part 4 first.")
else:
    palette  = ["#3498db", "#2ecc71", "#e67e22", "#e74c3c", "#9b59b6"]
    room_clr = dict(zip(R, palette))
    bar_clr  = [room_clr.get(r, "#bdc3c7") for r in sol["room_id"]]
    util_pct = (sol["enrolled"] / sol["capacity"] * 100).clip(upper=105)

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(range(len(sol)), util_pct, color=bar_clr, edgecolor="white", width=0.7)
    ax.axhline(100, color="red", linestyle="--", linewidth=1.2)
    ax.set_xticks(range(len(sol)))
    ax.set_xticklabels(sol["course"], rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("Utilisation (%)")
    ax.set_title("Room utilisation per course  (enrolled / room capacity)")
    patches = [mpatches.Patch(facecolor=room_clr[r], label=room_name[r]) for r in R]
    ax.legend(
        handles=[mpatches.Patch(color="red", label="100% full")] + patches,
        loc="upper right", fontsize=9
    )
    plt.tight_layout()
    plt.show()

    avg_util = util_pct.mean()
    wasted   = (sol["capacity"] - sol["enrolled"]).sum()
    print(f"Average utilisation : {avg_util:.1f}%")
    print(f"Total wasted seats  : {wasted}")
